# Visual Inspection & Pengecekan Akurasi Sampel GeoJSON Jawa Timur (5 Kelas)

Notebook ini digunakan untuk **memverifikasi secara visual** sampel poligon GeoJSON 5 kelas lahan Jawa Timur (`uts/data/geojson/jawatimur.geojson`) di atas **Citra Satelit Google (Google Satellite Basemap)** dan **Google Terrain (Medan)**.

Setiap kelas tutupan lahan diberi warna transparan khusus dan dapat di-toggle pada panel kontrol layer:
- 🌾 **Sawah / Pertanian** (Emas / Kuning)
- 🏠 **Pemukiman / Bangunan** (Oranye Merah)
- 🌊 **Hutan Mangrove** (Hijau Tua)
- 🌲 **Hutan Non-Mangrove** (Hijau Daun Hutan)
- 💧 **Air** (Biru Laut)

In [1]:
import os, geopandas as gpd, folium
from IPython.display import IFrame, display

geojson_path = "data/geojson/jawatimur.geojson"
html_out = "html/peta_cek_visual_geojson.html"

os.makedirs("html", exist_ok=True)

if os.path.exists(geojson_path):
    gdf = gpd.read_file(geojson_path)
    centroid = gdf.union_all().centroid
    map_center = [centroid.y, centroid.x]

    m = folium.Map(location=map_center, zoom_start=10, tiles=None)

    folium.TileLayer(
        tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
        attr="Google Satellite",
        name="Google Satellite Image",
        overlay=False,
        control=True
    ).add_to(m)

    folium.TileLayer(
        tiles="https://mt1.google.com/vt/lyrs=p&x={x}&y={y}&z={z}",
        attr="Google Terrain",
        name="Google Terrain (Medan)",
        overlay=False,
        control=True
    ).add_to(m)

    folium.TileLayer("openstreetmap", name="OpenStreetMap Standard").add_to(m)

    color_map = {
        "Sawah": "#FFD700",
        "Pemukiman": "#FF4500",
        "Hutan mangrove": "#006400",
        "Hutan non mangrove": "#228B22",
        "Air": "#1E90FF"
    }

    for class_name in gdf["Type"].unique():
        sub_gdf = gdf[gdf["Type"] == class_name]
        color = color_map.get(class_name, "#1E90FF")
        feature_group = folium.FeatureGroup(name=f"Kelas: {class_name} ({len(sub_gdf)} Poligon)")
        folium.GeoJson(
            sub_gdf.__geo_interface__,
            style_function=lambda feature, c=color: {
                "fillColor": c, "color": c, "weight": 2, "fillOpacity": 0.5
            },
            tooltip=folium.GeoJsonTooltip(fields=["Type"], aliases=["Kelas Lahan:"])
        ).add_to(feature_group)
        feature_group.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    m.save(html_out)
    print(f"SUCCESS: Peta Interaktif 5 Kelas 'jawatimur.geojson' disimpan di '{html_out}'")
    display(IFrame(html_out, width="100%", height=650))

SUCCESS: Peta Interaktif 5 Kelas 'jawatimur.geojson' disimpan di 'html/peta_cek_visual_geojson.html'
